CNN Pipeline

Typical flow:

Image
→ Conv
→ ReLU
→ Pool
→ Conv
→ ReLU
→ Pool
→ Flatten
→ FC layer
→ Prediction

This is standard CNN logic.

Most Important Insight

CNNs succeed because they:

preserve spatial structure->
reuse filters->
learn local patterns->
use far fewer weights

Normal NN:

Sees pixels as numbers

CNN:

Sees patterns and shapes

--------

Quick Summary

Problem with FC on images:

loses spatial information
too many weights

CNN solution:

convolution filters
local pattern detection
shared weights

Key components:

Layer	Purpose
Conv2D	detect patterns
ReLU	non-linearity
MaxPool	shrink + keep strongest signals
Flatten	prepare for FC layer

Main concept:

CNNs teach networks to see by scanning local image patterns instead of treating images as random lists of pixels.

In [1]:
# import torch
# import torchvision
# import torchvision.transforms as transforms
# import matplotlib.pyplot as plt
# import numpy as np

# # Normalise pixel values to mean=0.5, std=0.5 per channel
# transform = transforms.Compose([
#     transforms.ToTensor(),
#     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
# ])

# # Downloads automatically (~170MB, one time only)
# train_dataset = torchvision.datasets.CIFAR10(
#     root='./data', train=True,  download=True, transform=transform)
# val_dataset   = torchvision.datasets.CIFAR10(
#     root='./data', train=False, download=True, transform=transform)

# train_loader = torch.utils.data.DataLoader(
#     train_dataset, batch_size=64, shuffle=True,  num_workers=0)
# val_loader   = torch.utils.data.DataLoader(
#     val_dataset,   batch_size=64, shuffle=False, num_workers=0)

# classes = ['plane','car','bird','cat','deer',
#            'dog','frog','horse','ship','truck']

# print(f"Train batches: {len(train_loader)}")
# print(f"Val batches:   {len(val_loader)}")

In [2]:
# def imshow(img):
#     img = img / 2 + 0.5      # unnormalise
#     plt.imshow(np.transpose(img.numpy(), (1, 2, 0)))

# dataiter = iter(train_loader)
# images, labels = next(dataiter)

# plt.figure(figsize=(12, 3))
# for i in range(8):
#     plt.subplot(1, 8, i+1)
#     imshow(images[i])
#     plt.title(classes[labels[i]])
#     plt.axis('off')
# plt.tight_layout()
# plt.show()

# print("Image shape:", images[0].shape)  # [3, 32, 32] = channels, H, W

Big Picture

This step is:

Load
→ preprocess
→ batch
→ visualize
→ inspect shape

Exactly same philosophy as Week 1:

Understand data before modelling

Quick Summary

New concepts:

| Concept     | Meaning                    |
| ----------- | -------------------------- |
| torchvision | image utilities + datasets |
| transforms  | image preprocessing        |
| ToTensor    | image → tensor             |
| Normalize   | scale pixels               |
| CIFAR10     | image dataset              |
| DataLoader  | batch provider             |
| batch_size  | images per step            |
| shuffle     | randomize training order   |
| `[3,32,32]` | channels, height, width    |


Main idea:
Before training CNNs, you must preprocess, batch, and visually inspect image data so the network receives clean, correctly formatted inputs.

Input
[batch,3,32,32]

↓

Conv(3→32)
[batch,32,30,30]

↓

Pool
[batch,32,15,15]

↓

Conv(32→64)
[batch,64,13,13]

↓

Pool
[batch,64,6,6]

↓

Flatten
[batch,2304]

↓

FC
[batch,256]

↓

FC
[batch,10]

Real-Life Analogy

CNN pipeline:

Photo
→ edge detector
→ shape detector
→ compress information
→ summarize
→ classify

Like:

Police investigation
collect clues
compress evidence
final decision

Quick Summary

Layer effects:

Layer	Shape Effect
Conv	channels ↑, image slightly smaller
Pool	image shrinks
Flatten	3D → 1D
Linear	vector transformation

Key formulas:

Conv (no padding):

Output=Input−Kernel+1

Flatten:

Channels×Height×Width

Main concept:

CNN shape tracing is dimension bookkeeping — following how images transform through layers so you know exactly what size each layer receives and produces.

In [ ]:
# import torch.nn as nn
# import torch.nn.functional as F

# class CNN(nn.Module):
#     def __init__(self):
#         super().__init__()
#         # Convolutional layers
#         self.conv1 = nn.Conv2d(3, 32, kernel_size=3)   # 3 channels in, 32 out
#         self.conv2 = nn.Conv2d(32, 64, kernel_size=3)  # 32 in, 64 out
#         self.pool  = nn.MaxPool2d(2, 2)

#         # Fully connected layers
#         self.fc1 = nn.Linear(64 * 6 * 6, 256)  # 64 filters × 6×6 spatial
#         self.fc2 = nn.Linear(256, 10)            # 10 classes
#         self.dropout = nn.Dropout(0.3)

#     def forward(self, x):
#         x = self.pool(F.relu(self.conv1(x)))   # Conv → ReLU → Pool
#         x = self.pool(F.relu(self.conv2(x)))   # Conv → ReLU → Pool
#         x = x.view(x.size(0), -1)             # Flatten: [batch, 2304]
#         x = F.relu(self.fc1(x))
#         x = self.dropout(x)
#         x = self.fc2(x)                        # Raw scores (no softmax — CrossEntropyLoss handles it)
#         return x

# model = CNN()
# print(model)
# print("Params:", sum(p.numel() for p in model.parameters()))

CNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=2304, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=10, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)
Params: 612042


In [4]:
dummy = torch.randn(4, 3, 32, 32)   # batch of 4 images
out   = model(dummy)
print(out.shape)                     # should be [4, 10]

torch.Size([4, 10])


Quick Summary

New concepts:

Layer	Purpose
Conv2d	detect image patterns
MaxPool	shrink + keep strong signals
Flatten/view	3D → vector
Dropout	reduce overfitting
FC	classification reasoning

Important rules:

view(...,-1) → flatten
Dropout active only in train mode
CrossEntropyLoss → no softmax in model
Dummy input → shape debugging

Main concept:

A CNN learns visual patterns through Conv layers, compresses them with pooling, converts them into features, and finally classifies them using fully connected layers.

In [ ]:
# import torch.optim as optim

# device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model     = CNN().to(device)
# criterion = nn.CrossEntropyLoss()   # correct loss for multi-class
# optimizer = optim.Adam(model.parameters(), lr=0.001)

# train_losses, val_accuracies = [], []

# for epoch in range(10):
#     # Training
#     model.train()
#     running_loss = 0.0
#     for images, labels in train_loader:
#         images, labels = images.to(device), labels.to(device)

#         optimizer.zero_grad()
#         outputs = model(images)
#         loss    = criterion(outputs, labels)
#         loss.backward()
#         optimizer.step()

#         running_loss += loss.item()

#     avg_loss = running_loss / len(train_loader)
#     train_losses.append(avg_loss)

#     # Validation accuracy
#     model.eval()
#     correct, total = 0, 0
#     with torch.no_grad():
#         for images, labels in val_loader:
#             images, labels = images.to(device), labels.to(device)
#             outputs  = model(images)
#             _, preds = torch.max(outputs, 1)
#             correct += (preds == labels).sum().item()
#             total   += labels.size(0)

#     acc = correct / total
#     val_accuracies.append(acc)
#     print(f"Epoch {epoch+1:2d} | loss: {avg_loss:.4f} | val acc: {acc:.3f}")

Epoch  1 | loss: 1.4154 | val acc: 0.594
Epoch  2 | loss: 1.0669 | val acc: 0.649
Epoch  3 | loss: 0.9152 | val acc: 0.663
Epoch  4 | loss: 0.8123 | val acc: 0.703
Epoch  5 | loss: 0.7277 | val acc: 0.709
Epoch  6 | loss: 0.6514 | val acc: 0.718
Epoch  7 | loss: 0.5771 | val acc: 0.722
Epoch  8 | loss: 0.5188 | val acc: 0.727
Epoch  9 | loss: 0.4716 | val acc: 0.731
Epoch 10 | loss: 0.4165 | val acc: 0.730


Full Training Flow
Train mode
    ↓
Batch loop
    ↓
Forward
    ↓
Loss
    ↓
Backward
    ↓
Optimizer step
    ↓
Eval mode
    ↓
Validation accuracy
    ↓
Repeat epochs


Quick Summary

Important lines:

| Line               | Purpose                |
| ------------------ | ---------------------- |
| `.to(device)`      | move model/data        |
| `CrossEntropyLoss` | multi-class loss       |
| `zero_grad()`      | clear old gradients    |
| `backward()`       | compute gradients      |
| `step()`           | update weights         |
| `eval()`           | evaluation mode        |
| `no_grad()`        | faster inference       |
| `torch.max()`      | choose predicted class |


Main concept:
Training = repeatedly show batches, measure loss, compute gradients, update weights, then validate to check if learning generalises.

Image

↓

Conv
(find patterns)

↓

ReLU
(keep useful signals)

↓

Pool
(compress)

↓

Conv
(higher patterns)

↓

ReLU

↓

Pool

↓

Flatten
(make feature list)

↓

FC layer
(reason using clues)

↓

Prediction

CNN sees an image the way a human inspects a scene — first small patterns, then shapes, then objects.